# Part 8 — Deploy KNN Model to Azure ML Managed Online Endpoint

**Student:** Rukha Manahil

This notebook documents deployment of the registered `knn-maintenance-notebook` model
to the managed online endpoint `knn-maint-rukha21`, routing traffic to deployment `blue`,
and testing the endpoint with a sample request.


In [2]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "e23897b8-3ae4-41ac-b1c0-d291814c906b"
RESOURCE_GROUP = "rg-mlab1-rukha21"
WORKSPACE = "mlw-lab1-rukha21"

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE
)

print("Connected:", ml_client.workspace_name)

Connected: mlw-lab1-rukha21


## Deployment environment

The MLflow inference environment required `azureml-ai-monitoring`, so a corrected
Azure ML environment was created for the online deployment.


In [15]:
from azure.ai.ml.entities import Environment

fixed_env = Environment(
    name="knn-maintenance-deploy-fixed",
    description="MLflow inference environment with Azure ML monitoring dependency",
    image="mcr.microsoft.com/azureml/curated/mlflow-py312-inference@sha256:ccf97fa5140a75f616ec196d8e795035113ceaee731674fea437ae23e567a70a",
    conda_file={
        "name": "mlflow-env",
        "channels": ["conda-forge"],
        "dependencies": [
            "python=3.10.21",
            "pip<=26.2.1",
            {
                "pip": [
                    "azureml-inference-server-http",
                    "azureml-ai-monitoring",
                    "mlflow==3.15.0",
                    "numpy==1.26.4",
                    "pandas==2.3.3",
                    "psutil==7.2.2",
                    "pyarrow==25.0.1",
                    "scikit-learn==1.7.2",
                    "scipy==1.15.3",
                    "skops==0.16.0"
                ]
            }
        ]
    }
)

fixed_env = ml_client.environments.create_or_update(fixed_env)

print("Created:", fixed_env.name)
print("Version:", fixed_env.version)

Created: knn-maintenance-deploy-fixed
Version: 1


## Verify successful deployment

The final `blue` deployment uses the registered KNN model and the corrected environment.
The required MLflow model-folder environment variable was added during troubleshooting.
The cells below record the final successful deployment state and traffic configuration.


In [20]:
blue = ml_client.online_deployments.get(
    name="blue",
    endpoint_name="knn-maint-rukha21"
)

blue.environment_variables = {
    "AZUREML_MODEL_DIR": "/var/azureml-app/azureml-models/knn-maintenance-notebook/1",
    "MLFLOW_MODEL_FOLDER": "model_local"
}

ml_client.online_deployments.begin_create_or_update(
    blue
).result()

print("Blue deployment updated successfully.")

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint knn-maint-rukha21 exists


.............................................................Blue deployment updated successfully.


In [21]:
blue = ml_client.online_deployments.get(
    name="blue",
    endpoint_name="knn-maint-rukha21"
)

print("State:", blue.provisioning_state)
print("Environment variables:", blue.environment_variables)

State: DeploymentProvisioningState.SUCCEEDED
Environment variables: {'AZUREML_MODEL_DIR': '/var/azureml-app/azureml-models/knn-maintenance-notebook/1', 'MLFLOW_MODEL_FOLDER': 'model_local'}


In [22]:
endpoint = ml_client.online_endpoints.get(
    name="knn-maint-rukha21"
)

endpoint.traffic = {"blue": 100}

ml_client.online_endpoints.begin_create_or_update(
    endpoint
).result()

print("Traffic set to blue: 100%")

Traffic set to blue: 100%


## Create sample request

The request contains two machine observations using the same feature schema expected
by the deployed model.


In [23]:
import json
import os

sample = {
    "input_data": {
        "columns": [
            "air_temp_k",
            "process_temp_k",
            "rpm",
            "torque_nm",
            "tool_wear_min",
            "type"
        ],
        "index": [0, 1],
        "data": [
            [298.1, 308.6, 1551.0, 42.8, 0.0, "M"],
            [302.5, 310.9, 1325.0, 68.2, 215.0, "L"]
        ]
    }
}

os.makedirs("../deployment", exist_ok=True)

with open("../deployment/sample-request.json", "w") as f:
    json.dump(sample, f, indent=2)

print(json.dumps(sample, indent=2))

{
  "input_data": {
    "columns": [
      "air_temp_k",
      "process_temp_k",
      "rpm",
      "torque_nm",
      "tool_wear_min",
      "type"
    ],
    "index": [
      0,
      1
    ],
    "data": [
      [
        298.1,
        308.6,
        1551.0,
        42.8,
        0.0,
        "M"
      ],
      [
        302.5,
        310.9,
        1325.0,
        68.2,
        215.0,
        "L"
      ]
    ]
  }
}


## Invoke the online endpoint

A successful invocation should return predictions for the two sample machines.
`0` means OK and `1` means predicted machine failure.


In [24]:
response = ml_client.online_endpoints.invoke(
    endpoint_name="knn-maint-rukha21",
    deployment_name="blue",
    request_file="../deployment/sample-request.json"
)

print("Prediction (0 = OK, 1 = failure):", response)

Prediction (0 = OK, 1 = failure): [0, 1]


In [3]:
ml_client.online_endpoints.begin_delete(
    name="knn-maint-rukha21"
).result()

print("Endpoint deleted")

...................................................................................Endpoint deleted


In [2]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

print("MLClient connected")

Found the config file in: /config.json


MLClient connected
